In [1]:
using Random, Distributions, Statistics, Printf, DelimitedFiles, Dates
using LinearAlgebra
using StatsBase
using QuantileRegressions
using Plots
const bb = 27 
const aa = 38
const N  = 1462439
const I0 = 1
const S0 = 1316195
const n_iter = 1_000_000
include("functions.jl")
Random.seed!(2025)


Istar_obs = [3, 7, 10, 4, 14, 35, 92, 98, 216, 374, 434, 417, 447, 275, 151, 83, 67, 48, 37, 29, 42, 46, 53, 73, 87, 111, 123, 124, 99, 135, 106, 74, 39, 13]


tau = length(Istar_obs)

model_tag_sym = :powerlaw

KMAX_UPPER = 30  


# Fixed output dir
out_dir = "output"
isdir(out_dir) || mkpath(out_dir)

header_cont = []
if model_tag_sym === :memoryless
    global header_cont = ["beta", "alpha", "gamma"]
elseif model_tag_sym === :powerlaw
    global header_cont = ["beta", "alpha", "gamma", "lambda_P"]
elseif model_tag_sym === :exponential
    global header_cont = ["beta", "alpha", "gamma", "lambda_E"]
elseif model_tag_sym === :reciprocal
    global header_cont = ["beta", "alpha", "gamma", "lambda_R"] 
elseif model_tag_sym === :sliding
    global header_cont = ["beta", "alpha", "gamma"]            
else
    error("Unknown model tag: $(model_tag_sym)")
end

c = 3

@info "[$(String(model_tag_sym))_model] Fitting chain $(c) (tau=$tau)"

Random.seed!(2025 + c)
initθ_chain = initθ_for_chain(model_tag_sym) 
t0 = Dates.now()

try
    samples, loglik_aug_vecs =
        mcmc_one_chain_with_Rstar!(Istar_obs, N,S0, I0;
            fit_mech=model_tag_sym,
            n_iter=n_iter,
            initθ=initθ_chain,
            KMAX_UPPER=KMAX_UPPER)

    if size(samples, 1) != n_iter
        error("Chain $c did not complete all iterations.")
    end

    # Save samples
    samples_filename = "samples_chain_$(c).csv"
    write_csv(joinpath(out_dir, samples_filename), header_cont, samples)

    # Save per-time log-likelihoods (thinned & post-burnin inside mcmc)
    loglik_filename = "loglik_chain_$(c).csv"
    write_csv(joinpath(out_dir, loglik_filename), ["loglik"], hcat(loglik_aug_vecs))
    el = Dates.value(Dates.now() - t0) / 1000
catch err
    el = Dates.value(Dates.now() - t0) / 1000
end

@info "Chain completed -> output dir: $out_dir"


[ Info: [powerlaw_model] Fitting chain 3 (tau=34)
[ Info: [powerlaw] iter 1000/1000000 elapsed=3.9s, rate=0.128, mean=[1.632, 0.00095, 1.029, 0.496], std=[0.3207, 0.000396, 0.0183, 0.1164] [ADAPT]
[ Info: [powerlaw] iter 2000/1000000 elapsed=7.0s, rate=0.101, mean=[1.871, 0.00080, 1.022, 0.478], std=[0.3170, 0.000317, 0.0227, 0.0846] [ADAPT]
[ Info: [powerlaw] iter 3000/1000000 elapsed=9.2s, rate=0.090, mean=[1.968, 0.00075, 1.081, 0.485], std=[0.2988, 0.000272, 0.0873, 0.0718] [ADAPT]
[ Info: [powerlaw] iter 4000/1000000 elapsed=11.5s, rate=0.088, mean=[1.993, 0.00074, 1.142, 0.489], std=[0.2627, 0.000241, 0.1205, 0.0634] [ADAPT]
[ Info: [powerlaw] iter 5000/1000000 elapsed=13.7s, rate=0.086, mean=[2.018, 0.00073, 1.180, 0.490], std=[0.2408, 0.000220, 0.1283, 0.0571] [ADAPT]
[ Info: [powerlaw] iter 6000/1000000 elapsed=15.9s, rate=0.083, mean=[2.047, 0.00072, 1.221, 0.497], std=[0.2279, 0.000206, 0.1432, 0.0540] [ADAPT]
[ Info: [powerlaw] iter 7000/1000000 elapsed=18.2s, rate=0.083, m